# Notebook 05 — SARIMA Model
**CIS6035 IHMS — Somerset Mirissa Beach Hotel**

Steps:
1. Grid search seasonal order (P,D,Q,s=7) on top of best ARIMA order
2. Fit best SARIMA on train
3. Rolling 1-step forecast on validation
4. Compute RMSE, MAE, MAPE
5. Compare with ARIMA baseline
6. Save model artifact

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import joblib
import os
from itertools import product
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)
os.makedirs('../docs/figures', exist_ok=True)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))
def mae(y_true, y_pred):
    return np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

In [ ]:
train = pd.read_csv('../data/splits/train.csv', index_col='date', parse_dates=True)['occupancy_rate']
val   = pd.read_csv('../data/splits/val.csv',   index_col='date', parse_dates=True)['occupancy_rate']
test  = pd.read_csv('../data/splits/test.csv',  index_col='date', parse_dates=True)['occupancy_rate']

# Load ARIMA best order for non-seasonal component
arima_artifact = joblib.load('../models/arima_model.pkl')
arima_order = arima_artifact['order']
print(f'Using ARIMA non-seasonal order: {arima_order}')

In [ ]:
# SARIMA seasonal grid search (P, D, Q) with s=7
S = 7
P_range = range(0, 3)  # 0..2
D_range = range(0, 2)  # 0..1
Q_range = range(0, 3)  # 0..2

best_aic = np.inf
best_seasonal = (1, 1, 0)
seasonal_results = []

print(f'Grid searching SARIMA{arima_order}(P,D,Q,{S})...')
for P, D, Q in product(P_range, D_range, Q_range):
    try:
        model = SARIMAX(
            train,
            order=arima_order,
            seasonal_order=(P, D, Q, S),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fitted = model.fit(disp=False)
        seasonal_results.append({'P': P, 'D': D, 'Q': Q, 'aic': fitted.aic})
        if fitted.aic < best_aic:
            best_aic = fitted.aic
            best_seasonal = (P, D, Q)
    except Exception:
        pass

sr_df = pd.DataFrame(seasonal_results).sort_values('aic')
print(f'Best seasonal order: {best_seasonal}  AIC: {best_aic:.2f}')
sr_df.head()

In [ ]:
# Fit best SARIMA
sarima_model = SARIMAX(
    train,
    order=arima_order,
    seasonal_order=(*best_seasonal, S),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

print(sarima_model.summary())

In [ ]:
# Rolling 1-step forecast on validation
print('Rolling forecast on validation...')
history = list(train)
val_preds = []

for i, actual in enumerate(val):
    model = SARIMAX(
        history,
        order=arima_order,
        seasonal_order=(*best_seasonal, S),
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    pred = float(np.clip(model.forecast(steps=1)[0], 0, 1))
    val_preds.append(pred)
    history.append(actual)
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{len(val)}')

val_rmse = rmse(val, val_preds)
val_mae  = mae(val, val_preds)
val_mape = mape(val, val_preds)

print(f'\nSARIMA{arima_order}{best_seasonal},{S} Validation Metrics:')
print(f'  RMSE: {val_rmse:.4f}')
print(f'  MAE:  {val_mae:.4f}')
print(f'  MAPE: {val_mape:.2f}%')

In [ ]:
# Plot
arima_art = joblib.load('../models/arima_model.pkl')

plt.figure(figsize=(16, 5))
plt.plot(val.index, val.values, label='Actual', color='#0f172a', linewidth=1)
plt.plot(val.index, val_preds, label=f'SARIMA Forecast', color='#8b5cf6',
         linewidth=1.5, linestyle='--')
plt.title('SARIMA — Rolling Forecast on Validation Set (2024)', fontsize=13, fontweight='bold')
plt.ylabel('Occupancy Rate')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/figures/05_sarima_val_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save artifact
sarima_artifact = {
    'model': sarima_model,
    'order': arima_order,
    'seasonal_order': (*best_seasonal, S),
    'val_rmse': val_rmse,
    'val_mae': val_mae,
    'val_mape': val_mape,
    'train_end': str(train.index.max().date()),
}
joblib.dump(sarima_artifact, '../models/sarima_model.pkl')
print('SARIMA model saved to models/sarima_model.pkl')